# 整備済みの13表を実際に見る

**基本市場表1＋案A4条件＋需要3表＋計画需給2表＋連系線3表＝13表**を確認できます。
保存形式はParquet（型を保った表ファイル）です。単一のDBサーバーではなく、目的・粒度ごとに分けた表です。

1. repoの `.venv` をカーネルに選び、「すべて実行」してください。
2. 各表の見出しの下に、行数・列数・主要列の先頭10行が表示されます。
3. 最後の「自由に閲覧」で、表・地域・日付・表示列・ページを切り替えられます。
   「全列」は元列・出典・QA列も含み、横にスクロールして見られます。

表示範囲を絞るのは閲覧のためだけです。原ファイルの削除・補完・書換え・集約は行いません。
案A4条件は代替シナリオです。最終baselineの選択や4倍の市場観測としての合算は行いません。

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import HTML, display

# worktreeからも、実データのある共有repoを探す。
relative_data = Path("data/processed/occto/demand_supply_direction/20261016_native_v1")
repo = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "AGENTS.md").exists() and (p / relative_data).is_dir()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError(
        "共有repoのdata/が見つかりません。このrepo内から実行してください。"
        "GitHubからのcloneだけでは実データは含まれません。"
    )
data_dir = repo / relative_data
print(f"読取先: {data_dir}")

In [ ]:
TABLES = {
    "market": {"title": "基本市場表", "filename": "market_base_FY2025_with_reserve.parquet"},
    "a_58852_early": {
        "title": "案A：58,852行snapshot・早端",
        "filename": "market_hjks_outage_58852_early_FY2025_with_reserve.parquet",
    },
    "a_58852_late": {
        "title": "案A：58,852行snapshot・遅端",
        "filename": "market_hjks_outage_58852_late_FY2025_with_reserve.parquet",
    },
    "a_7828_early": {
        "title": "案A：7,828行snapshot・早端",
        "filename": "market_hjks_outage_7828_early_FY2025_with_reserve.parquet",
    },
    "a_7828_late": {
        "title": "案A：7,828行snapshot・遅端",
        "filename": "market_hjks_outage_7828_late_FY2025_with_reserve.parquet",
    },
    "demand_hourly": {"title": "需要：1時間", "filename": "occto_demand_hourly_FY2025.parquet"},
    "demand_energy": {
        "title": "需要：日電力量",
        "filename": "occto_demand_daily_energy_FY2025.parquet",
    },
    "demand_peak": {
        "title": "需要：日最大電力",
        "filename": "occto_demand_daily_peak_FY2025.parquet",
    },
    "reserve_nextday2": {
        "title": "計画需給：翌々日",
        "filename": "occto_reserve_nextday2_FY2025.parquet",
    },
    "reserve_latest": {
        "title": "計画需給：翌日・当日",
        "filename": "occto_reserve_latest_FY2025.parquet",
    },
    "network_system": {
        "title": "連系線：系統空容量・翌々日策定",
        "filename": "system_capacity_dayahead2_planned_with_directions_FY2025.parquet",
    },
    "network_nextday2": {
        "title": "連系線：予備率・翌々日",
        "filename": "reserve_interconnector_nextday2_with_directions_FY2025.parquet",
    },
    "network_latest": {
        "title": "連系線：予備率・翌日当日",
        "filename": "reserve_interconnector_latest_with_directions_FY2025.parquet",
    },
}

market_columns = [
    "area",
    "delivery_date",
    "period",
    "area_price",
    "system_price_market",
    "system_price_sensitivity_base",
    "baseline_price_match_flag",
    "reserve_nextday2__area_demand_forecast_mw",
    "reserve_latest__area_supply_plan_mw",
]
annotation_columns = [
    "annotation__observed_unit_count",
    "annotation__reference_capacity_known_kw",
    "annotation__reference_gap_partial_sum_kw",
    "annotation__missing_output_count",
    "annotation__capacity_hold_count",
    "annotation__negative_gap_count",
    "annotation__matched_planned_stop__observed_unit_count",
    "annotation__no_applicable_record__observed_unit_count",
]


def main_columns(key):
    if key == "market" or key.startswith("a_"):
        return market_columns + (annotation_columns if key.startswith("a_") else [])
    if key.startswith("demand_"):
        value = {
            "demand_hourly": "demand_hourly_csv_mwh",
            "demand_energy": "demand_daily_energy_mwh",
            "demand_peak": "demand_daily_peak_mw",
        }[key]
        return [
            "area",
            "delivery_date",
            "area_scope",
            value,
            "quantity_unit",
            "qa_hourly_unit_display_conflict",
        ] + (
            ["hour_of_delivery", "timestamp_start", "timestamp_end"]
            if key == "demand_hourly"
            else []
        )
    if key.startswith("reserve_"):
        return [
            "area",
            "delivery_date",
            "period",
            "area_demand_forecast_mw",
            "area_supply_plan_mw",
            "area_reserve_published_mw",
            "wide_area_reserve_rate_published_pct",
            "block_number_raw",
            "reserve_information_stage",
            "file_latest_updated_at",
        ]
    if key == "network_system":
        return [
            "delivery_date",
            "period",
            "link_name_raw",
            "direction_raw",
            "sending_area_set_json",
            "receiving_area_set_json",
            "available_capacity_mw",
            "operating_capacity_mw",
            "planned_flow_mw",
            "direction_status",
        ]
    flow = "forecast" if key == "network_nextday2" else "planned"
    return [
        "delivery_date",
        "period",
        "link_name_raw",
        "forward__sending_area_set_json",
        "forward__receiving_area_set_json",
        "forward_available_capacity_mw",
        "reverse_available_capacity_mw",
        f"forward_{flow}_flow_mw",
        "forward__direction_status",
        "equipment_scope_class",
    ]


inventory = []
for key, info in TABLES.items():
    stored = pq.ParquetFile(data_dir / info["filename"])
    inventory.append(
        {
            "選択ID": key,
            "表": info["title"],
            "行数": stored.metadata.num_rows,
            "列数": len(stored.schema_arrow),
            "ファイル": info["filename"],
        }
    )
assert len(inventory) == 13
display(pd.DataFrame(inventory))

## 閲覧用の共通関数

`read_page` は指定した表を**全列で**読み、日付・地域等で絞ってページ表示します。
`columns="main"`なら主要列、`columns=None`なら全列、列名リストなら任意の列です。
連系線は「地域」の列がないため、地域の代わりに元の連系線名を指定します。
フィルターできない列はエラーとして知らせ、条件を黙って無視しません。

In [ ]:
def read_page(
    key, *, area=None, delivery_date=None, period=None, link=None, offset=0, limit=20, columns=None
):
    if offset < 0 or limit < 1:
        raise ValueError("開始行は0以上、表示件数は1以上にしてください。")
    dataset = ds.dataset(data_dir / TABLES[key]["filename"], format="parquet")
    selected_columns = main_columns(key) if columns == "main" else columns
    expression = None
    for field, value in [
        ("area", area),
        ("delivery_date", delivery_date),
        ("period", period),
        ("link_name_raw", link),
    ]:
        if value is None or value == "":
            continue
        if field not in dataset.schema.names:
            raise ValueError(f"この表に{field}はありません。該当フィルターを空にしてください。")
        if field == "delivery_date" and isinstance(value, str):
            value = date.fromisoformat(value)
        condition = ds.field(field) == pa.scalar(value, type=dataset.schema.field(field).type)
        expression = condition if expression is None else expression & condition
    matched = dataset.count_rows(filter=expression)
    chunks = []
    skipped = 0
    remaining = limit
    for batch in dataset.to_batches(columns=selected_columns, filter=expression, batch_size=2048):
        if skipped + batch.num_rows <= offset:
            skipped += batch.num_rows
            continue
        begin = max(0, offset - skipped)
        part = batch.slice(begin, min(remaining, batch.num_rows - begin))
        chunks.append(part)
        remaining -= part.num_rows
        skipped += batch.num_rows
        if remaining == 0:
            break
    schema = (
        dataset.schema
        if selected_columns is None
        else pa.schema([dataset.schema.field(c) for c in selected_columns])
    )
    frame = pa.Table.from_batches(chunks, schema=schema).to_pandas()
    frame.attrs["matched_rows"] = matched
    frame.attrs["offset"] = offset
    return frame


def show_frame(frame):
    print(
        f"条件に合う全{frame.attrs['matched_rows']:,}行のうち、"
        f"{frame.attrs['offset']:,}行目から{len(frame):,}行を表示 / {len(frame.columns)}列"
    )
    display(
        HTML(
            '<div style="overflow:auto;max-height:480px">'
            + frame.to_html(index=False, max_rows=None, max_cols=None)
            + "</div>"
        )
    )


def show_table(key):
    info = TABLES[key]
    stored = pq.ParquetFile(data_dir / info["filename"])
    print(f"{info['filename']}\n{stored.metadata.num_rows:,}行・{len(stored.schema_arrow)}列")
    fields = pd.DataFrame([{"列名": f.name, "型": str(f.type)} for f in stored.schema_arrow])
    display(
        HTML(
            "<details><summary>全列名と型を開く</summary>"
            '<div style="overflow:auto;max-height:360px">'
            + fields.to_html(index=False, escape=True)
            + "</div></details>"
        )
    )
    frame = read_page(key, limit=10, columns="main")
    show_frame(frame)
    return frame

## 1．基本市場表

地域固有の`area_price`と全国共通のsystem price・感応度を区別します。
spotと感応度の基準価格は2,271時刻で異なるままです。全国共通値は地域へ反復されます。

In [ ]:
previews = {}
previews["market"] = show_table("market")

## 2．案Aの市場表：4条件をそれぞれ見る

snapshotは停止原本58,852行版と7,828行版です。早端・遅端は復旧予定日の00:00／翌日00:00という予定区間の診断シナリオです。
主区分は境界保留を維持するため差が出ないことがあっても、区間該当flagは変わります。
参照容量差は対象設備の部分和で、地域全体の余力ではありません。記録なしは停止なしを意味しません。
以下の4表は相互に代替する条件です。

### 案A：58,852行snapshot・早端

基本市場表＋参照容量差の部分和と公表記録区分。実停止認定や停止損失控除ではありません。

In [ ]:
previews["a_58852_early"] = show_table("a_58852_early")

### 案A：58,852行snapshot・遅端

基本市場表＋参照容量差の部分和と公表記録区分。実停止認定や停止損失控除ではありません。

In [ ]:
previews["a_58852_late"] = show_table("a_58852_late")

### 案A：7,828行snapshot・早端

基本市場表＋参照容量差の部分和と公表記録区分。実停止認定や停止損失控除ではありません。

In [ ]:
previews["a_7828_early"] = show_table("a_7828_early")

### 案A：7,828行snapshot・遅端

基本市場表＋参照容量差の部分和と公表記録区分。実停止認定や停止損失控除ではありません。

In [ ]:
previews["a_7828_late"] = show_table("a_7828_late")

## 3．需要：元頻度の3表

### 需要：1時間

1時間×10エリアと10エリア総計。CSV仕様上MWh、画面説明との相違をflagで保持。30分へ補間しません。

In [ ]:
previews["demand_hourly"] = show_table("demand_hourly")

### 需要：日電力量

日×10エリアと総計。単位MWh。総計と個別エリアを加算しません。

In [ ]:
previews["demand_energy"] = show_table("demand_energy")

### 需要：日最大電力

日×10エリアと総計。単位MW。発生時刻は補っていません。

In [ ]:
previews["demand_peak"] = show_table("demand_peak")

## 4．計画需給：2断面

### 計画需給：翌々日

30分×10エリア。需要は予測、供給力は計画、予備率は公表値。ブロック共通値の地域反復を足しません。

In [ ]:
previews["reserve_nextday2"] = show_table("reserve_nextday2")

### 計画需給：翌日・当日

30分×10エリア。latestは取得snapshotの計画断面で、実績需要・実発電ではありません。

In [ ]:
previews["reserve_latest"] = show_table("reserve_latest")

## 5．連系線：3系列

### 連系線：系統空容量・翌々日策定

30分×連系線×順逆。数量はMW。逆方向の原負符号を保ちます。

In [ ]:
previews["network_system"] = show_table("network_system")

### 連系線：予備率・翌々日

30分×連系線、原1行に両方向列。予想潮流と方向保留を保持。

In [ ]:
previews["network_nextday2"] = show_table("network_nextday2")

### 連系線：予備率・翌日当日

30分×連系線、原1行に両方向列。計画潮流は実潮流ではありません。

In [ ]:
previews["network_latest"] = show_table("network_latest")

## 6．自由に閲覧：全13表・全列・任意の日付

表を選び、必要なら地域・日付（例`2025-08-01`）・period・連系線名を入力して「表示」を押します。
空欄は絞り込みなしです。前後のページも表示できます。行順は保存済みの順序で、日付順とは限りません。
「全列」で省略せず横スクロールできます。ページ送りは閲覧上の区切りで、未表示行は元表に残ります。

DataFrameとして調べる場合は下の`selected_page`、各表のプレビューは`previews`にあります。
これらは表示範囲だけです。全行をメモリへ読みたい場合は最後の例を明示的に実行してください。

In [ ]:
# ipywidgetsが使えない環境でも、次のコードセルで全表・全列を閲覧できます。
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("対話欄が使えないため、次のセルの選択ID・条件を編集して閲覧してください。")

if widgets is not None:
    table_choice = widgets.Dropdown(
        options=[(info["title"], key) for key, info in TABLES.items()],
        description="表",
        layout=widgets.Layout(width="650px"),
    )
    area_input = widgets.Text(description="地域", placeholder="空欄＝全地域。例：東京")
    date_input = widgets.Text(description="受渡日", placeholder="YYYY-MM-DD または空欄")
    period_input = widgets.Text(description="period", placeholder="1〜48 または空欄")
    link_input = widgets.Text(description="連系線", placeholder="元の名称と完全一致 または空欄")
    view_choice = widgets.ToggleButtons(options=[("全列", "all"), ("主要列", "main")])
    row_offset = widgets.IntText(value=0, description="開始行")
    page_size = widgets.BoundedIntText(value=20, min=1, max=500, description="表示件数")
    show_button = widgets.Button(description="表示", button_style="primary")
    previous_button = widgets.Button(description="前のページ")
    next_button = widgets.Button(description="次のページ")
    viewer_output = widgets.Output()

    def draw_page(_=None):
        global selected_page
        with viewer_output:
            viewer_output.clear_output(wait=True)
            try:
                selected_page = read_page(
                    table_choice.value,
                    area=area_input.value.strip() or None,
                    delivery_date=date_input.value.strip() or None,
                    period=int(period_input.value) if period_input.value.strip() else None,
                    link=link_input.value.strip() or None,
                    offset=row_offset.value,
                    limit=page_size.value,
                    columns=None if view_choice.value == "all" else "main",
                )
                show_frame(selected_page)
            except (ValueError, KeyError, pa.ArrowException) as error:
                selected_page = None
                print(f"表示条件を確認してください: {error}")

    def change_page(step):
        row_offset.value = max(0, row_offset.value + step * page_size.value)
        draw_page()

    def reset_filters(_):
        area_input.value = ""
        date_input.value = ""
        period_input.value = ""
        link_input.value = ""
        row_offset.value = 0

    table_choice.observe(reset_filters, names="value")
    show_button.on_click(draw_page)
    previous_button.on_click(lambda _: change_page(-1))
    next_button.on_click(lambda _: change_page(1))
    display(
        widgets.VBox(
            [
                table_choice,
                widgets.HBox([area_input, date_input]),
                widgets.HBox([period_input, link_input]),
                view_choice,
                widgets.HBox([row_offset, page_size]),
                widgets.HBox([show_button, previous_button, next_button]),
                viewer_output,
            ]
        )
    )
    draw_page()

### コードから閲覧する場合

選択IDは冒頭の一覧にあります。このセルの値を変えて実行すれば、対話欄が使えない場合も閲覧できます。
日集計にperiod、連系線にareaを指定することはできません。

In [ ]:
TABLE_KEY = "market"  # 例：a_7828_late / demand_hourly / network_system
AREA = None  # 例："東京"。連系線ではNone
DELIVERY_DATE = None  # 例："2026-03-31"
PERIOD = None  # 例：48。1時間需要・日集計ではNone
LINK = None  # 例：元の連系線名
OFFSET = 0
LIMIT = 20
COLUMNS = None  # None＝全列 / "main"＝主要列 / ["area", ...]＝指定列

manual_page = read_page(
    TABLE_KEY,
    area=AREA,
    delivery_date=DELIVERY_DATE,
    period=PERIOD,
    link=LINK,
    offset=OFFSET,
    limit=LIMIT,
    columns=COLUMNS,
)
show_frame(manual_page)

### 任意の1表を全行読み込む場合

上の表示はページ単位です。全行・全列が必要な場合だけ、次の例を新しいコードセルで実行してください。
約定価格や欠測の修正はせず、読み込むだけです。案Aなど列の多い表はメモリを多く使います。

```python
key = "demand_hourly"  # 読みたい表の選択ID
full_table = pq.read_table(data_dir / TABLES[key]["filename"]).to_pandas()
display(full_table)  # VS Codeでは変数full_tableをデータビューアーで開けます
```

13表は原本から整備・接続した現在の成果物で、最終研究baselineを選んだDBではありません。
需要は1時間/日、需給は予測・計画、案Aはsnapshot記録、連系線はリンク別公表値という意味を保っています。
定義・単位・既知の注意は[成果物ガイド](../docs/data_guide.md)と[変数辞書](../metadata/variable_dictionary.csv)を参照してください。